In [1]:
from pyspark.sql import SparkSession, functions as F

spark = SparkSession.builder.appName("kedaibuku").getOrCreate()

LAKE = "../data/lake"

orders = spark.read.parquet(f"{LAKE}/silver/orders.parquet")
books = spark.read.parquet(f"{LAKE}/silver/books.parquet")

sales = (
    orders.join(
        books.select("book_id", "category", "price_myr"),
        "book_id"
    )
    .withColumn(
        "revenue_myr",
        F.round(F.col("quantity") * F.col("price_myr"), 2)
    )
    .withColumn(
        "order_month",
        F.date_format("order_date", "yyyy-MM")
    )
)

# Write one folder per month
sales.write.mode("overwrite") \
    .partitionBy("order_month") \
    .parquet(f"{LAKE}/gold/sales_by_month")

june = (
    spark.read.parquet(f"{LAKE}/gold/sales_by_month")
    .filter(F.col("order_month") == "2026-06")
)

print(june.count())
june.explain()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/11/26 02:52:45 WARN Utils: Your hostname, codespaces-561287, resolves to a loopback address: 127.0.0.1; using 10.0.20.103 instead (on interface eth0)
26/11/26 02:52:45 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/26 02:52:48 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


367
== Physical Plan ==
*(1) ColumnarToRow
+- FileScan parquet [book_id#18,order_id#19,customer_id#20,quantity#21L,order_date#22,category#23,price_myr#24,revenue_myr#25,order_month#26] Batched: true, DataFilters: [], Format: Parquet, Location: InMemoryFileIndex(1 paths)[file:/workspaces/kedaibuku-de/data/lake/gold/sales_by_month], PartitionFilters: [isnotnull(order_month#26), (order_month#26 = 2026-06)], PushedFilters: [], ReadSchema: struct<book_id:string,order_id:string,customer_id:string,quantity:bigint,order_date:timestamp_ntz...


